# Chapter 20 — Intelligent Troubleshooting Agents
## From Theory to the NOC

It is 2 AM. Your phone buzzes. "Internet is down at headquarters."
You groan, grab the laptop, and start the familiar dance — SSH, `show ip route`,
`show ip bgp summary`, one command at a time.

Now imagine a different scenario: one sentence to a chat interface,
and two minutes later the agent returns the exact root cause and the fix.
It ran twelve diagnostics while you were still looking for your coffee.

This chapter explains **why** that is possible, and shows you how to build it.

| Technique | What It Does | Why It Matters |
|-----------|-------------|----------------|
| **Chain of Thought (CoT)** | Forces the LLM to write out every reasoning step | Prevents "statistical leaps" to wrong answers |
| **Tree of Thoughts (ToT)** | Explores multiple hypotheses in parallel | Mirrors how senior engineers actually think |
| **Self-Consistency** | Runs the same problem 3× and votes | Majority rules out individual hallucinations |
| **Coordinator-Worker** | Splits tasks across specialized agents | Each agent is an expert at one thing |
| **Full NOC Pipeline** | Combines everything for P1 response | Production-grade automated troubleshooting |

> **No GPU needed.** All demos use the Anthropic API (Haiku for sub-tasks, Sonnet for reasoning).

### Install
```
pip install anthropic
```


## Setup — API Client and Shared Utilities

In [ ]:
import os, json, re, time
from anthropic import Anthropic

try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    import getpass
    if not os.environ.get('ANTHROPIC_API_KEY'):
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')
client = Anthropic()   # reads ANTHROPIC_API_KEY from env

HAIKU  = "claude-haiku-4-5-20251001"   # fast — for sub-tasks and workers
SONNET = "claude-sonnet-4-20250514"           # smart — for reasoning and coordination

def ask(prompt, model=HAIKU, system="You are a network troubleshooting expert.",
        temperature=0.0):
    """Single-turn helper — returns plain text."""
    r = client.messages.create(
        model=model,
        max_tokens=1024,
        system=system,
        messages=[{"role": "user", "content": prompt}],
    )
    return r.content[0].text.strip()

# Simulated network state — what diagnostic commands would return
NETWORK_STATE = {
    "show_ip_bgp_summary":  "Neighbor 10.0.0.1: Idle (Admin), Uptime 00:00:00, Prefixes 0",
    "show_ip_route":        "Gateway of last resort is not set. No route to 203.0.113.0/24",
    "show_ip_interface":    "GigabitEthernet0/0: up/up, GigabitEthernet0/1: up/up",
    "show_run_bgp":         "neighbor 10.0.0.1 route-map BLOCK-OUT out",
    "show_route_map":       "route-map BLOCK-OUT deny 10 — match ip address prefix-list BLOCK",
    "show_prefix_list":     "ip prefix-list BLOCK: seq 5 deny 0.0.0.0/0 le 32 (matches all!)",
    "show_access_list":     "ACL 100: permit tcp any any eq 179",
    "show_ip_ospf_neighbor":"No OSPF neighbors found",
    "show_log":             "BGP: neighbor 10.0.0.1 went from Active to Idle, route-map denied update",
    "show_cpu":             "CPU utilization: 8% — normal",
}

def run_tool(command: str) -> str:
    """Simulates running a network diagnostic command."""
    key = command.lower().replace(" ", "_").replace("-", "_")
    for k, v in NETWORK_STATE.items():
        if k in key or key in k:
            return v
    return f"Command '{command}' output: (no specific data — check manually)"

print("Client ready.")
print(f"Diagnostic tools: {len(NETWORK_STATE)} simulated commands available")


---
## Demo 1 — Chain of Thought (CoT): Make the Agent Write Its Reasoning

Without CoT, an LLM makes a single "statistical leap" from symptom to answer.
It might get lucky, but it often guesses wrong — especially for multi-step problems.

**Chain of Thought** forces the model to write every reasoning step as output.
Each step becomes context that guides the next — like a math student showing their work.

```
Without CoT:  "User can't reach internet" → "Check BGP" (lucky guess)

With CoT:     Step 1: "Only one user prefix affected → L1/L2 probably fine"
              Step 2: "No route to destination → routing issue"
              Step 3: "BGP session is Idle → check policy"
              Step 4: "route-map BLOCK-OUT → check what it matches"
              Answer: "Route-map is blocking all prefixes — bug in prefix-list"
```

CoT works because: each intermediate step is a *token* in the context window,
constraining the next prediction. The model can't skip steps.


In [ ]:
# ── Chain of Thought Troubleshooting ─────────────────────────────────────────

def cot_troubleshoot(symptom: str, tools_available: list[str]) -> str:
    """
    Uses explicit CoT prompting to guide step-by-step reasoning.
    The key: we ask for numbered steps BEFORE the final answer.
    """
    prompt = f"""Network problem: {symptom}

Available diagnostic commands: {tools_available}

Work through this STEP BY STEP. Do not jump to a conclusion.

For each step:
1. State what you know so far
2. Decide which command gives the most information
3. Interpret the output

Use this format:
STEP 1: [reasoning] → run: [command]
OUTPUT: [what the command returns]
INTERPRETATION: [what this tells you]

STEP 2: [reasoning] → run: [command]
...

FINAL ANSWER: [root cause + fix]
"""
    # We run this with the model — it generates all steps
    response = ask(prompt, model=SONNET)
    return response


def run_interactive_cot(symptom: str):
    """
    More realistic version: agent requests tools one at a time.
    We execute each tool and feed the result back.
    """
    print(f"Problem: {symptom}")
    print("=" * 60)

    messages = [
        {"role": "user", "content": f"""Network problem: {symptom}

Troubleshoot step by step. For each step, output EXACTLY:
THINKING: [your reasoning]
TOOL: [one command to run]

After receiving each output, continue reasoning.
When you have enough evidence, output:
CONCLUSION: [root cause]
FIX: [specific commands to resolve it]

Start with step 1."""}
    ]

    for step in range(1, 8):   # max 7 steps
        # Get next reasoning step from model
        resp = client.messages.create(
            model=SONNET,
            max_tokens=512,
            system="You are a senior network engineer. Think step by step.",
            messages=messages,
        )
        agent_text = resp.content[0].text.strip()
        print(f"\n[Step {step}]")
        print(agent_text)

        # Check if agent reached a conclusion
        if "CONCLUSION:" in agent_text or "FIX:" in agent_text:
            print("\n✓ Root cause identified.")
            break

        # Extract tool call
        tool_match = re.search(r"TOOL:\s*(.+)", agent_text)
        if not tool_match:
            break
        command = tool_match.group(1).strip()

        # Run the tool (simulated)
        tool_output = run_tool(command)
        print(f"\n  [Tool Output] {command}:")
        print(f"  → {tool_output}")

        # Feed result back to agent
        messages.append({"role": "assistant", "content": agent_text})
        messages.append({"role": "user",
                         "content": f"Command output: {tool_output}\n\nContinue reasoning."})


run_interactive_cot("Users at HQ cannot reach the internet. BGP session shows Idle.")


---
## Demo 2 — Tree of Thoughts (ToT): Chase Multiple Hypotheses at Once

CoT is linear — one reasoning path, one conclusion.
But real troubleshooting isn't linear. An experienced engineer keeps
**multiple hypotheses alive** simultaneously.

**Tree of Thoughts** adds branching:
1. Generate 3 candidate hypotheses from the symptom
2. Evaluate each hypothesis with evidence
3. Prune branches that the evidence contradicts
4. Continue only on the surviving branches

```
Symptom: "BGP session down"
        │
   ┌────┼────┐
   A    B    C
  Auth MTU  Route-map
   │    │    │
 ✓OK  ✓OK  ✗MATCH → root cause!
        │
    (pruned)
```

This is how senior engineers think. They don't commit to one theory —
they hold the possibility space open until evidence forces a decision.


In [ ]:
# ── Tree of Thoughts for Network Diagnostics ─────────────────────────────────

class ThoughtNode:
    def __init__(self, hypothesis: str, confidence: float, evidence: list[str]):
        self.hypothesis  = hypothesis
        self.confidence  = confidence    # 0.0 – 1.0
        self.evidence    = evidence      # list of supporting/refuting facts
        self.pruned      = False

def tot_troubleshoot(symptom: str):
    print(f"Symptom: {symptom}")
    print("=" * 60)

    # ── Phase 1: Generate candidate hypotheses ────────────────────────────────
    gen_prompt = f"""Network symptom: {symptom}

List exactly 3 distinct, specific hypotheses for the root cause.
For each, estimate initial probability (0-100%).
Format:
H1: <hypothesis> | Probability: <N>%
H2: <hypothesis> | Probability: <N>%
H3: <hypothesis> | Probability: <N>%
"""
    raw = ask(gen_prompt, model=SONNET)
    print("\n[Phase 1] Candidate hypotheses:")
    print(raw)

    # Parse hypotheses
    hypotheses = []
    for line in raw.splitlines():
        m = re.match(r"H\d:\s*(.+?)\s*\|\s*Probability:\s*(\d+)%", line)
        if m:
            hypotheses.append(ThoughtNode(m.group(1), int(m.group(2))/100, []))

    # ── Phase 2: Gather evidence for each hypothesis ──────────────────────────
    print("\n[Phase 2] Gathering evidence per branch:")
    evidence_map = {
        "auth":       run_tool("show_run_bgp"),
        "route_map":  run_tool("show_route_map"),
        "prefix":     run_tool("show_prefix_list"),
        "bgp":        run_tool("show_ip_bgp_summary"),
        "interface":  run_tool("show_ip_interface"),
        "log":        run_tool("show_log"),
    }

    for i, node in enumerate(hypotheses):
        eval_prompt = f"""Hypothesis: {node.hypothesis}

Available evidence:
{json.dumps(evidence_map, indent=2)}

Evaluate this hypothesis:
1. Which evidence supports it? (cite specific data)
2. Which evidence contradicts it?
3. Updated probability (0-100%): give a number only at the end as "PROBABILITY: N%"
4. Should this branch be PRUNED or KEPT?
"""
        evaluation = ask(eval_prompt, model=SONNET)
        node.evidence.append(evaluation)

        # Extract updated probability
        prob_match = re.search(r"PROBABILITY:\s*(\d+)%", evaluation)
        if prob_match:
            node.confidence = int(prob_match.group(1)) / 100

        # Prune if probability drops very low
        node.pruned = "PRUNED" in evaluation.upper() or node.confidence < 0.15

        status = "✗ PRUNED" if node.pruned else "✓ ALIVE"
        print(f"\n  H{i+1}: {node.hypothesis[:60]}...")
        print(f"  Confidence: {node.confidence:.0%}  Status: {status}")

    # ── Phase 3: Conclude from surviving branches ─────────────────────────────
    surviving = [n for n in hypotheses if not n.pruned]
    print(f"\n[Phase 3] Surviving branches: {len(surviving)}")

    if not surviving:
        print("All branches pruned — need more evidence.")
        return

    best = max(surviving, key=lambda n: n.confidence)
    conclude_prompt = f"""Best hypothesis: {best.hypothesis}
Confidence: {best.confidence:.0%}

Evidence summary:
{best.evidence[0][:500] if best.evidence else 'None'}

Write a 3-sentence conclusion:
1. Root cause (specific)
2. Evidence that confirms it
3. Exact fix
"""
    conclusion = ask(conclude_prompt, model=SONNET)
    print(f"\nRoot Cause (ToT consensus):")
    print(conclusion)


tot_troubleshoot("BGP session to upstream ISP is in Idle state. No internet connectivity.")


---
## Demo 3 — Self-Consistency: Run 3 Paths, Vote on the Answer

Even the best reasoning can contain errors. A single CoT run might:
- Misinterpret a command output
- Pick the wrong tool
- Hallucinate a conclusion

**Self-Consistency** solves this by running the *same problem* through
3 independent reasoning paths (with slightly varied temperature) and
selecting the conclusion that the **majority agrees on**.

```
Path 1: ... → "route-map is blocking prefixes"
Path 2: ... → "route-map is blocking prefixes"
Path 3: ... → "authentication mismatch"

Vote: route-map wins 2-1 → confident answer
```

This is like asking three senior engineers to troubleshoot the same issue
independently, then comparing notes. Where two or more agree, that's the answer.


In [ ]:
# ── Self-Consistency Voting ───────────────────────────────────────────────────

def single_reasoning_path(symptom: str, path_id: int, temperature: float) -> str:
    """One independent reasoning run. Temperature adds variation."""
    # Gather key evidence
    evidence = {
        "bgp_summary": run_tool("show ip bgp summary"),
        "route_map":   run_tool("show route-map"),
        "prefix_list": run_tool("show prefix-list"),
        "log":         run_tool("show log"),
    }

    prompt = f"""Network problem: {symptom}

Diagnostic evidence:
{json.dumps(evidence, indent=2)}

Reason through this and state the root cause in ONE SENTENCE starting with:
ROOT_CAUSE: [your conclusion]

Then give one fix command:
FIX: [command]
"""
    resp = client.messages.create(
        model=HAIKU,
        max_tokens=256,
        system="You are a network engineer. Be concise and specific.",
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.content[0].text.strip()


def self_consistency_vote(symptom: str, n_paths: int = 3):
    print(f"Problem: {symptom}")
    print(f"Running {n_paths} independent reasoning paths...")
    print("=" * 60)

    # Run N independent paths
    conclusions = []
    for i in range(1, n_paths + 1):
        # Slightly vary temperature for independence
        temp = 0.0 + (i - 1) * 0.3
        result = single_reasoning_path(symptom, i, temp)

        # Extract root cause
        rc_match = re.search(r"ROOT_CAUSE:\s*(.+)", result)
        root_cause = rc_match.group(1).strip() if rc_match else result[:100]
        conclusions.append(root_cause)
        print(f"\nPath {i} (temp={temp:.1f}):")
        print(f"  → {root_cause}")

    # ── Vote using LLM to cluster similar answers ──────────────────────────────
    print("\n[Voting Phase]")
    vote_prompt = f"""Three independent reasoning paths reached these conclusions:

Path 1: {conclusions[0]}
Path 2: {conclusions[1]}
Path 3: {conclusions[2]}

Do they agree on the root cause?
1. Group similar conclusions together
2. Which conclusion has the strongest majority support?
3. State the final agreed root cause in ONE sentence
4. Confidence: HIGH / MEDIUM / LOW

Format your answer as:
AGREEMENT: <yes/no/partial>
FINAL: <conclusion>
CONFIDENCE: <HIGH/MEDIUM/LOW>
"""
    verdict = ask(vote_prompt, model=SONNET)
    print(verdict)

    return verdict


self_consistency_vote(
    "Internet access down at HQ. BGP shows Idle. No route to 0.0.0.0/0.",
    n_paths=3
)


---
## Demo 4 — Coordinator-Worker Multi-Agent NOC Team

A single agent has one perspective. A **NOC team** has specialists:
- **L1 Analyst** — runs basic checks, decides severity
- **L2 Network Engineer** — deep dives into routing and protocols
- **Security Analyst** — looks for policy and ACL issues
- **Coordinator** — assigns work, synthesizes findings, gives final answer

This is the **Coordinator-Worker model**: the coordinator never touches a tool.
The workers never communicate with users. Each agent is expert at one thing.

```
        ┌─ Coordinator ─┐
        │    (Sonnet)   │
        │ assigns tasks │
        └───────────────┘
               │
    ┌──────────┼──────────┐
    ▼          ▼          ▼
  L1 NOC    L2 NetEng  Security
  (Haiku)   (Haiku)    (Haiku)
  basic     routing    policy
  checks    analysis   audit
    └──────────┼──────────┘
               ▼
        Final Report
```


In [ ]:
# ── Coordinator-Worker NOC Team ───────────────────────────────────────────────

# Shared incident state — workers update this dict
incident_state = {}

# ── Worker Agents ─────────────────────────────────────────────────────────────

def l1_analyst(incident: str) -> dict:
    """L1 NOC: basic health checks, severity triage."""
    # L1 always runs the same first-response checklist
    checks = {
        "interface_status": run_tool("show ip interface"),
        "cpu":              run_tool("show cpu"),
        "bgp_summary":      run_tool("show ip bgp summary"),
    }
    prompt = f"""L1 NOC triage for: {incident}

Basic checks:
{json.dumps(checks, indent=2)}

Provide:
SEVERITY: P1/P2/P3
AFFECTED: what is affected
ESCALATE_TO: L2_NETENG or L2_SECURITY or BOTH
FINDINGS: 2 bullet points
"""
    result = ask(prompt, model=HAIKU,
                 system="You are a L1 NOC analyst. Quick triage only.")
    return {"agent": "L1_NOC", "findings": result, "raw_data": checks}


def l2_network_engineer(incident: str, l1_findings: str) -> dict:
    """L2 Network Engineer: deep routing and protocol analysis."""
    checks = {
        "routing_table": run_tool("show ip route"),
        "bgp_detail":    run_tool("show ip bgp summary"),
        "route_map":     run_tool("show route-map"),
        "prefix_list":   run_tool("show prefix-list"),
        "logs":          run_tool("show log"),
    }
    prompt = f"""L2 Network Engineering analysis.

Incident: {incident}
L1 Findings: {l1_findings}

Deep diagnostic data:
{json.dumps(checks, indent=2)}

Identify:
ROOT_CAUSE: [specific technical cause]
EVIDENCE: [3 specific data points that prove it]
FIX: [exact CLI commands]
"""
    result = ask(prompt, model=HAIKU,
                 system="You are a L2 network engineer. Precise technical analysis.")
    return {"agent": "L2_NETENG", "findings": result, "raw_data": checks}


def security_analyst(incident: str, l1_findings: str) -> dict:
    """Security: checks for policy violations, ACLs, route-maps."""
    checks = {
        "acl":          run_tool("show access-list"),
        "route_map":    run_tool("show route-map"),
        "prefix_list":  run_tool("show prefix-list"),
        "bgp_policy":   run_tool("show run bgp"),
    }
    prompt = f"""Security policy audit.

Incident: {incident}
L1 Context: {l1_findings}

Policy data:
{json.dumps(checks, indent=2)}

Report:
POLICY_ISSUES: [any misconfigurations or overly broad rules]
SECURITY_RISK: LOW/MEDIUM/HIGH
RECOMMENDATION: [specific fix]
"""
    result = ask(prompt, model=HAIKU,
                 system="You are a network security analyst. Focus on policy and ACLs.")
    return {"agent": "SECURITY", "findings": result, "raw_data": checks}


# ── Coordinator ───────────────────────────────────────────────────────────────

def coordinator(incident: str):
    """
    Coordinator orchestrates the team:
    1. Send to L1 for triage
    2. Based on L1 findings, dispatch L2 workers in parallel
    3. Synthesize all findings into one report
    """
    print(f"INCIDENT: {incident}")
    print("=" * 60)

    # Step 1: L1 Triage
    print("\n[Coordinator] Dispatching to L1 NOC...")
    l1_result = l1_analyst(incident)
    print(f"\n[L1 NOC] Findings:")
    print(l1_result["findings"])

    # Step 2: Dispatch specialists based on L1 escalation
    print("\n[Coordinator] Escalating to L2 specialists...")
    l2_result   = l2_network_engineer(incident, l1_result["findings"])
    sec_result  = security_analyst(incident, l1_result["findings"])

    print(f"\n[L2 NetEng] Findings:")
    print(l2_result["findings"])
    print(f"\n[Security] Findings:")
    print(sec_result["findings"])

    # Step 3: Coordinator synthesizes all findings
    print("\n[Coordinator] Synthesizing final report...")
    synth_prompt = f"""You are the NOC coordinator. Synthesize these findings into a final incident report.

L1 Triage: {l1_result['findings']}
L2 Network Engineering: {l2_result['findings']}
Security Audit: {sec_result['findings']}

Write a brief NOC report:
INCIDENT SUMMARY: (1 sentence)
ROOT CAUSE: (specific)
CONTRIBUTING FACTORS: (bullet list)
RESOLUTION STEPS: (numbered, ordered)
LESSONS LEARNED: (1 sentence)
"""
    final_report = ask(synth_prompt, model=SONNET,
                       system="You are a NOC coordinator writing a post-incident report.")
    print("\n" + "=" * 60)
    print("FINAL NOC REPORT")
    print("=" * 60)
    print(final_report)


coordinator("Users at HQ cannot reach the internet. BGP to upstream ISP shows Idle state.")


---
## Demo 5 — Full NOC Pipeline: All Techniques for a P1 Incident

Production intelligent troubleshooting agents don't pick one technique —
they compose them in layers:

```
P1 Alert: "Internet down at HQ"
    │
    ├─ [CoT] Step-by-step initial reasoning → narrows to BGP
    │
    ├─ [ToT] Three branches: auth / route-map / MTU → evidence prunes 2
    │
    ├─ [Self-Consistency] 3 paths vote → confirms route-map root cause
    │
    └─ [Multi-Agent NOC] L1 + L2 + Security → final report + fix
```

Each layer adds confidence. By the time the coordinator writes the report,
the root cause has been confirmed by independent reasoning paths, branch pruning,
and specialist agents — not just one lucky guess.


In [ ]:
# ── Full Production NOC Pipeline ─────────────────────────────────────────────

def full_noc_pipeline(alert: str):
    print(f"P1 ALERT: {alert}")
    print("=" * 70)
    start = time.time()
    pipeline_log = []

    # ── Stage 1: CoT Initial Assessment ──────────────────────────────────────
    print("\n[Stage 1] CoT — Initial step-by-step assessment")
    cot_prompt = f"""Network alert: {alert}

Run a step-by-step assessment. Use these diagnostic results:
- BGP: {run_tool('show ip bgp summary')}
- Routes: {run_tool('show ip route')}
- Interfaces: {run_tool('show ip interface')}

STEP 1: What do the interfaces tell us?
STEP 2: What does the routing table tell us?
STEP 3: What does the BGP state tell us?
INITIAL_HYPOTHESIS: [one sentence]
"""
    cot_result = ask(cot_prompt, model=SONNET)
    # Extract hypothesis
    hyp_match = re.search(r"INITIAL_HYPOTHESIS:\s*(.+)", cot_result)
    initial_hypothesis = hyp_match.group(1) if hyp_match else "BGP policy issue suspected"
    print(f"  Initial hypothesis: {initial_hypothesis}")
    pipeline_log.append(f"CoT hypothesis: {initial_hypothesis}")

    # ── Stage 2: ToT Branch Evaluation ───────────────────────────────────────
    print("\n[Stage 2] ToT — Branch evaluation")
    tot_prompt = f"""Hypothesis: {initial_hypothesis}

Evaluate 3 branches using this evidence:
- Route-map: {run_tool('show route-map')}
- Prefix-list: {run_tool('show prefix-list')}
- BGP config: {run_tool('show run bgp')}
- Logs: {run_tool('show log')}

For each branch, give probability 0-100%:
BRANCH_A: Authentication mismatch | P=?%
BRANCH_B: Route-map policy block  | P=?%
BRANCH_C: MTU/TCP issue           | P=?%
WINNER: [branch name]
"""
    tot_result = ask(tot_prompt, model=SONNET)
    winner_match = re.search(r"WINNER:\s*(.+)", tot_result)
    winning_branch = winner_match.group(1).strip() if winner_match else "Route-map policy block"
    print(f"  Winning branch: {winning_branch}")

    # Extract probabilities for display
    for branch in ["BRANCH_A", "BRANCH_B", "BRANCH_C"]:
        m = re.search(rf"{branch}:.+?P=(\d+)%", tot_result)
        prob = m.group(1) if m else "?"
        print(f"  {branch}: {prob}%")
    pipeline_log.append(f"ToT winner: {winning_branch}")

    # ── Stage 3: Self-Consistency Confirmation ────────────────────────────────
    print("\n[Stage 3] Self-Consistency — 3-path vote")
    all_evidence = {
        "bgp":    run_tool("show ip bgp summary"),
        "routes": run_tool("show ip route"),
        "policy": run_tool("show route-map"),
        "prefix": run_tool("show prefix-list"),
        "logs":   run_tool("show log"),
    }
    votes = []
    for path_n in range(1, 4):
        path_prompt = f"""Alert: {alert}
Working hypothesis: {winning_branch}
Evidence: {json.dumps(all_evidence)}

State root cause in one sentence starting with ROOT_CAUSE:
Then state fix starting with FIX:
"""
        path_result = ask(path_prompt, model=HAIKU)
        rc = re.search(r"ROOT_CAUSE:\s*(.+)", path_result)
        votes.append(rc.group(1).strip() if rc else path_result[:80])
        print(f"  Path {path_n}: {votes[-1][:70]}...")
    pipeline_log.append(f"Self-consistency votes: {len(votes)} paths agreed")

    # ── Stage 4: Multi-Agent Final Report ────────────────────────────────────
    print("\n[Stage 4] Multi-Agent synthesis — final NOC report")
    report_prompt = f"""Generate a concise NOC incident report for:

Alert: {alert}
CoT finding: {initial_hypothesis}
ToT confirmed: {winning_branch}
3-path consensus: {votes[0] if votes else 'route-map policy block'}

Report format:
SEVERITY: P1
ROOT CAUSE: (one sentence — specific)
EVIDENCE: (3 bullet points with specific data)
IMMEDIATE FIX:
  1. (exact command)
  2. (exact command)
VERIFICATION: (command to confirm fix worked)
PREVENTION: (one sentence)
"""
    final_report = ask(report_prompt, model=SONNET,
                       system="You are a NOC coordinator writing a concise incident report.")

    elapsed = time.time() - start
    print("\n" + "=" * 70)
    print("FINAL NOC INCIDENT REPORT")
    print("=" * 70)
    print(final_report)
    print("\n" + "=" * 70)
    print(f"Pipeline complete in {elapsed:.1f}s")
    print(f"Stages: CoT → ToT ({winning_branch}) → Self-Consistency (3 paths) → Multi-Agent Report")
    print("=" * 70)


full_noc_pipeline("Internet outage at HQ. All users affected. BGP to upstream ISP in Idle state.")


---
## Summary — Building Your NOC Agent

### The Reasoning Stack

| Layer | Technique | What It Adds |
|-------|-----------|-------------|
| 1 | **CoT** | Traceable step-by-step logic — no more lucky guesses |
| 2 | **ToT** | Parallel hypothesis exploration — prune wrong branches early |
| 3 | **Self-Consistency** | Majority voting — catches individual hallucinations |
| 4 | **Multi-Agent** | Specialist depth — each agent is expert at one domain |

### Key Principles

1. **Temperature = 0** for troubleshooting. Creativity is the enemy of diagnostic accuracy.
2. **CoT before action.** Never let the agent jump straight to a fix command.
3. **Prune aggressively.** A ToT branch below 20% probability wastes resources.
4. **Workers are cheap.** Use Haiku for L1/L2 workers. Sonnet only for coordination and final synthesis.
5. **Tool descriptions matter more than tool code.** The LLM selects tools by reading descriptions — be specific about what each tool returns and when to use it.

### The 2 AM Scenario — Revisited

With this pipeline:
- **CoT** narrows the problem from "internet is down" to "BGP policy issue" in 3 steps
- **ToT** confirms the route-map branch with 85% confidence
- **Self-consistency** has 3 independent paths agree on the same fix
- **Multi-agent** provides L2-quality analysis with a verifiable fix command

**Total time: ~90 seconds.** While you're still looking for your coffee.

### What's Next

- **Chapter 21**: Production Safety — guardrails, circuit breakers, and human-in-the-loop approval gates before any agent touches a production network

> The gap between a "demo agent" and a "2 AM agent" is the reasoning stack. CoT + ToT + Self-Consistency is that stack.
